# Runtime-scaling scan planner

Generates a midpoint-convention BubbleMaster GPU scan from three physical
inputs: the gamma_* range and the kR_* range and N_omega. Grid spacing
(`dgamma_ij`/`dt`) and resolution/numerics (`wall_points`/`N_min`/
`panels_per_oscillation`/`how_often_ds`/`s_batch_size`) are locked
production values baked into `ptbuilder.ic.generate_surrogate` -- not
exposed as scan inputs here (see that function for the validated values).

Re-running this notebook with a different `(GAMMA_STAR_MIN, GAMMA_STAR_MAX,
KR_MIN, KR_MAX, N_OMEGA)` writes a separate, distinctly-named scan under
`data/runtime_scan_.../` and `scripts/runtime_scan_...sh` -- so several scans
(e.g. increasing `GAMMA_STAR_MAX`) can be run and timed side by side to see
how total runtime scales with the domain size.

**Extending an existing scan**: set `EXTEND_FROM` to an existing
`runtime_scan_.../` directory and raise `GAMMA_STAR_MAX` above what it was
built with (up to `OMEGA_MIN_GAMMA_FACTOR` times the old value -- the
frequency-grid headroom baked in when it was built). This reads the old
scan's actual gamma_ij grid and locked parameters back, extends every
existing row's `times[]` in place to the new, larger `T_MAX`
(`ptbuilder.ic.extend_2d_setup_times` -- wlist/geometry untouched), adds any
new `gamma_ij` rows the larger range needs, and renames the scan directory
to match the new `GAMMA_STAR_MAX`. Re-running `bubblemaster_gpu` on the
(now-extended) setups auto-detects and extends each row from its own
embedded plateau checkpoint -- no separate resume step needed.

In [ ]:
from pathlib import Path

import ptbuilder as pt
from ptbuilder.ic import generate_surrogate
from ptbuilder.physics import Phi4Potential, PhysicsModel

ROOT = Path(pt.__file__).resolve().parent.parent
LAMBDA_BAR = 0.84
TAG_LABEL = f'lb{LAMBDA_BAR:g}'

model = PhysicsModel(Phi4Potential(LAMBDA_BAR))

## Scan inputs

Only these three define the physical scan domain.

In [2]:
GAMMA_STAR_MIN = 1.0
GAMMA_STAR_MAX = 9.6
KR_MIN = 1.0
KR_MAX = 100.0
N_OMEGA = 32

# Optional overrides -- production defaults, only change these deliberately.
PAIR_GAMMA_FACTOR   = 2.5   # gamma_ij,max = PAIR_GAMMA_FACTOR * GAMMA_STAR_MAX
TIME_FACTOR         = 1.4   # T_MAX = TIME_FACTOR * R_*(GAMMA_STAR_MAX)
OMEGA_MIN_GAMMA_FACTOR = 1.5   # OMEGA_MIN anchored to R_*(this * GAMMA_STAR_MAX),
                               # giving headroom to later extend GAMMA_STAR_MAX by
                               # up to this factor without an empty omega range

# Set to an existing runtime_scan_.../ directory to EXTEND that scan to the
# (larger) GAMMA_STAR_MAX above, instead of building a fresh scan. When set,
# GAMMA_STAR_MIN/KR_MIN/KR_MAX/PAIR_GAMMA_FACTOR/TIME_FACTOR/
# OMEGA_MIN_GAMMA_FACTOR are read back from the old scan and OVERRIDE the
# values above -- only GAMMA_STAR_MAX (the new target) is actually used from
# this cell. Leave as None for a normal, from-scratch scan.
EXTEND_FROM = None

## GPU account

Grid spacing and resolution/numerics are locked production values defined
in `ptbuilder.ic.generate_surrogate`, not exposed here.

In [ ]:
GPU_ACCOUNT = 'm3166_g'

In [ ]:
scan = generate_surrogate(
    model, ROOT, TAG_LABEL, GAMMA_STAR_MIN, GAMMA_STAR_MAX, KR_MIN, KR_MAX, N_OMEGA,
    extend_from=EXTEND_FROM, pair_gamma_factor=PAIR_GAMMA_FACTOR,
    time_factor=TIME_FACTOR, omega_min_gamma_factor=OMEGA_MIN_GAMMA_FACTOR,
)

## SLURM script

Runtimes across the grid vary by 2-3 orders of magnitude (seconds at low
gamma_ij up to hours near GAMMA_IJ_MAX), so sizing the allocation for all
`n_tasks` at once (one node per 4 tasks, all launched together) leaves most
GPUs idle waiting for the few slow, high-gamma_ij pairs to finish.

Instead this requests a small, fixed-size pool (`GPU_NODES` nodes x 4
GPUs/node) and streams the manifest through it with `xargs -P`: as soon as
one `srun` finishes, the next pending row starts on that freed GPU slot.

In [ ]:
GPU_WALLTIME = '04:00:00'   # set generously -- no walltime estimate here (see intro)
GPU_NODES    = 2            # size of the worker pool; concurrency = GPU_NODES * 4
FORCE_OVERWRITE = False     # pass --overwrite to every row -- ignore/discard any existing
                            # results or checkpoint and recompute from scratch (e.g. to
                            # regenerate an old scan with the current binary)

def write_slurm_script(scan, gpu_walltime='04:00:00', gpu_nodes=2, force_overwrite=False,
                       gpu_account='m3166_g'):
    """Writes scripts/runtime_scan_{tag}_gpu.sh for `scan` (from
    generate_surrogate) -- duplicated in notebook 17 too rather than shared,
    since this is cluster-specific (SLURM syntax, account, node counts)."""
    (ROOT / 'logs').mkdir(exist_ok=True)
    gpus_per_node = 4
    n_concurrent  = gpu_nodes * gpus_per_node
    n_tasks       = len(scan.manifest_rows)
    relative_manifest = scan.manifest.relative_to(ROOT)
    relative_output   = scan.output_root.relative_to(ROOT)
    overwrite_flag = ' --overwrite' if force_overwrite else ''

    script_path = ROOT / 'scripts' / f'runtime_scan_{scan.tag}_gpu.sh'
    script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{scan.tag}_gpu
#SBATCH --output=logs/bm_{scan.tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {gpu_account}
#SBATCH --nodes={gpu_nodes}
#SBATCH --ntasks={n_concurrent}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={gpu_walltime}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"

run_one() {{
    local row_num=$1
    local line
    line=$(sed -n "$((row_num + 1))p" {relative_manifest})
    IFS=$'\t' read -r index gamma n_t setup name <<< "$line"
    output="{relative_output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {scan.n_min} --filon-panels-per-osc {scan.panels_per_oscillation} --s-batch-size {scan.s_batch_size}{overwrite_flag}
}}
export -f run_one

seq 1 {n_tasks} | xargs -P {n_concurrent} -I ROW bash -c 'run_one "$@"' _ ROW
''')
    script_path.chmod(0o755)
    print(f'Written: {script_path.relative_to(ROOT)}  '
          f'({n_tasks} tasks, {n_concurrent} concurrent slots on {gpu_nodes} nodes)'
          + (', --overwrite' if force_overwrite else ''))
    return script_path


script_path = write_slurm_script(
    scan, gpu_walltime=GPU_WALLTIME, gpu_nodes=GPU_NODES,
    force_overwrite=FORCE_OVERWRITE, gpu_account=GPU_ACCOUNT,
)
print(f'Submit: sbatch {script_path.relative_to(ROOT)}')